# CLASS 7
# Machine Learning Package - Hyperparameter Optimization

In [1]:
import numpy as np

from si.data.dataset import Dataset
from si.metrics.nn_accuracy import nn_accuracy
from si.model_selection.grid_search import grid_search_cv
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.layers import DenseLayer
from si.neural_networks.losses import BinaryCrossEntropy
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.optimizers import SGD

## Exercise — Tune the NeuralNetwork with grid_search_cv
**Builder function: creates a fresh, untrained network for each fold (slide 10).**

In [2]:
def build_network(learning_rate, n_hidden):
    np.random.seed(0)   # same initial weights for every configuration
    net = NeuralNetwork(epochs        = 100, 
                        batch_size    = 16, 
                        optimizer     = SGD,
                        learning_rate = learning_rate, 
                        momentum      = 0.9,
                        loss          = BinaryCrossEntropy, 
                        metric        = nn_accuracy)
    net.add(DenseLayer(n_hidden, (2,)))
    net.add(ReLUActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    return net

**Dataset: is the point inside the circle? Grid search over learning_rate and n_hidden.**

In [3]:
rng = np.random.default_rng(42)
X   = rng.uniform(-2, 2, size = (400, 2))
y   = (X[:, 0]**2 + X[:, 1]**2 < 1.5).astype(float)    # inside the circle?
ds  = Dataset(X, y)

grid    = {'learning_rate': [0.001, 0.01, 0.1], 'n_hidden': [2, 4, 16]}
results = grid_search_cv(build_network, ds, grid, cv = 5, seed = 1)

for h, s in zip(results['hyperparameters'], results['scores']):
    print(h, round(s, 3))

print(results['best_hyperparameters'], results['best_score'])

{'learning_rate': 0.001, 'n_hidden': 2} 0.715
{'learning_rate': 0.001, 'n_hidden': 4} 0.715
{'learning_rate': 0.001, 'n_hidden': 16} 0.843
{'learning_rate': 0.01, 'n_hidden': 2} 0.727
{'learning_rate': 0.01, 'n_hidden': 4} 0.722
{'learning_rate': 0.01, 'n_hidden': 16} 0.975
{'learning_rate': 0.1, 'n_hidden': 2} 0.713
{'learning_rate': 0.1, 'n_hidden': 4} 0.765
{'learning_rate': 0.1, 'n_hidden': 16} 0.98
{'learning_rate': 0.1, 'n_hidden': 16} 0.9800000000000001


**Last question: pass a network already built (instead of the builder) and search over learning_rate.**

In [4]:
results_model = grid_search_cv(build_network(0.01, 16), ds, {'learning_rate': [0.0001, 0.1]}, cv = 5, seed = 1)

for h, s in zip(results_model['hyperparameters'], results_model['scores']):
    print(h, round(s, 3))

{'learning_rate': 0.0001} 0.975
{'learning_rate': 0.1} 0.982


**Observations:**

- **Best configuration:** learning_rate = 0.1 and n_hidden = 16, with a mean validation accuracy of 0.98.
- **Why do several configurations give 0.715?** 71.5% of the samples (286 of 400) are outside the circle (class 0). 0.715 is the accuracy of a network that always predicts class 0: with a small learning rate or too few hidden neurons, the network does not learn the circular boundary and stays at the majority class.
- **How many models were trained?** 9 configurations × 5 folds = 45 models.
- **Passing build_network(0.01, 16) with {'learning_rate': [0.0001, 0.1]}:** the scores barely change (0.975 and 0.982), and both are close to the score of learning_rate = 0.01 with 16 neurons (0.975). With learning_rate = 0.0001 the network should not learn at all. The change of learning_rate is ignored: the optimizer is created in the NeuralNetwork __init__ and copied into the layers by add, so setattr(net, 'learning_rate', ...) after that changes nothing. The small difference between the two scores comes from the random shuffle of the mini-batches, not from the learning rate. This is why a builder function is needed.

## Exercise 17 — randomized_search_cv
**log_uniform: samples are spread evenly over the orders of magnitude.**

In [5]:
from si.model_selection.randomized_search import randomized_search_cv, log_uniform

sampler = log_uniform(1e-4, 1e-1)
rng_lu  = np.random.default_rng(0)
samples = np.array([sampler(rng_lu) for _ in range(1000)])

print('Min:', samples.min(), ' Max:', samples.max())
for low in (1e-4, 1e-3, 1e-2):
    print(f'[{low:g}, {low * 10:g}[: {np.mean((samples >= low) & (samples < low * 10)):.2f}')

Min: 0.00010013133462910044  Max: 0.09965613890316331
[0.0001, 0.001[: 0.30
[0.001, 0.01[: 0.34
[0.01, 0.1[: 0.36


**Randomized search on the circle dataset (same build_network as before).**

In [6]:
distributions = {'learning_rate': log_uniform(1e-4, 1e-1), 'n_hidden': [2, 4, 16]}

results_random = randomized_search_cv(build_network, ds, distributions, cv = 5, n_iter = 10, seed = 1)

for h, s in zip(results_random['hyperparameters'], results_random['scores']):
    print({'learning_rate': round(h['learning_rate'], 5), 'n_hidden': h['n_hidden']}, round(s, 3))

print(results_random['best_hyperparameters'], round(results_random['best_score'], 3))

{'learning_rate': 0.0693, 'n_hidden': 4} 0.772
{'learning_rate': 0.00112, 'n_hidden': 16} 0.857
{'learning_rate': 0.00016, 'n_hidden': 16} 0.715
{'learning_rate': 0.00032, 'n_hidden': 16} 0.718
{'learning_rate': 0.00448, 'n_hidden': 2} 0.66
{'learning_rate': 0.0047, 'n_hidden': 16} 0.965
{'learning_rate': 0.00188, 'n_hidden': 4} 0.715
{'learning_rate': 0.00533, 'n_hidden': 4} 0.723
{'learning_rate': 0.00237, 'n_hidden': 16} 0.925
{'learning_rate': 0.03254, 'n_hidden': 4} 0.747
{'learning_rate': 0.004701393908419611, 'n_hidden': 16} 0.965


## Exercise 18 — Tune the neural network of Exercise 16
**Dataset of Exercise 16 (32 features, binary classification), split into train and test (test_size = 0.2). The search uses only the training set.**

In [7]:
from si.model_selection.split import train_test_split

np.random.seed(42)

n_samples, n_features = 500, 32

X_32   = np.random.randn(n_samples, n_features)
true_w = np.random.randn(n_features)
y_32   = (X_32 @ true_w > 0).astype(float)

ds_32 = Dataset(X_32, y_32)

train_32, test_32 = train_test_split(ds_32, test_size = 0.2, random_state = 42)

print('Train:', train_32.shape())
print('Test: ', test_32.shape())

Train: (400, 32)
Test:  (100, 32)


**Builder function with the hyperparameters learning_rate, batch_size and epochs (architecture of Exercise 16).**

In [8]:
def build_network_16(learning_rate, batch_size, epochs):
    np.random.seed(0)   # same initial weights for every configuration
    net = NeuralNetwork(epochs        = epochs, 
                        batch_size    = batch_size, 
                        optimizer     = SGD,
                        learning_rate = learning_rate,
                        loss          = BinaryCrossEntropy, 
                        metric        = nn_accuracy)
    net.add(DenseLayer(n_features // 2, (n_features,)))   # 32 -> 16
    net.add(ReLUActivation())
    net.add(DenseLayer(n_features // 4))                  # 16 -> 8
    net.add(ReLUActivation())
    net.add(DenseLayer(1))                                # output
    net.add(SigmoidActivation())
    return net

**Randomized search: n_iter = 20, cv = 3, scoring = accuracy.**

In [9]:
distributions_16 = {'learning_rate': log_uniform(1e-4, 1e-1),
                    'batch_size':    [8, 16, 32, 64],
                    'epochs':        [50, 100, 200]}

results_16 = randomized_search_cv(build_network_16, train_32, distributions_16, 
                                  scoring = nn_accuracy, cv = 3, n_iter = 20, seed = 42)

for h, s in zip(results_16['hyperparameters'], results_16['scores']):
    print({'learning_rate': round(h['learning_rate'], 5), 'batch_size': h['batch_size'], 'epochs': h['epochs']}, round(s, 4))

print()
print('Best hyperparameters:', results_16['best_hyperparameters'])
print('Best CV accuracy:    ', round(results_16['best_score'], 4))

{'learning_rate': 0.00025, 'batch_size': 64, 'epochs': 200} 0.8325
{'learning_rate': 0.00349, 'batch_size': 32, 'epochs': 200} 0.905
{'learning_rate': 0.00245, 'batch_size': 64, 'epochs': 200} 0.905
{'learning_rate': 0.00829, 'batch_size': 16, 'epochs': 50} 0.9025
{'learning_rate': 0.00026, 'batch_size': 16, 'epochs': 100} 0.7075
{'learning_rate': 0.00178, 'batch_size': 16, 'epochs': 100} 0.9
{'learning_rate': 0.00127, 'batch_size': 8, 'epochs': 100} 0.9025
{'learning_rate': 0.00096, 'batch_size': 16, 'epochs': 100} 0.895
{'learning_rate': 0.01141, 'batch_size': 16, 'epochs': 50} 0.9075
{'learning_rate': 0.07024, 'batch_size': 16, 'epochs': 200} 0.9175
{'learning_rate': 0.00277, 'batch_size': 64, 'epochs': 200} 0.9025
{'learning_rate': 0.00404, 'batch_size': 16, 'epochs': 200} 0.905
{'learning_rate': 0.00907, 'batch_size': 64, 'epochs': 50} 0.9075
{'learning_rate': 0.00396, 'batch_size': 64, 'epochs': 200} 0.905
{'learning_rate': 0.00073, 'batch_size': 32, 'epochs': 50} 0.7575
{'learni

**Refit the best configuration on the whole training set and evaluate on the test set; compare with the hyperparameters of Exercise 16.**

In [10]:
best_net = build_network_16(**results_16['best_hyperparameters'])
best_net.fit(train_32)

ex16_net = build_network_16(learning_rate = 0.01, batch_size = 16, epochs = 100)
ex16_net.fit(train_32)

print(f'Tuned (best configuration) test accuracy: {best_net.score(test_32):.4f}')
print(f'Exercise 16 hyperparameters test accuracy: {ex16_net.score(test_32):.4f}')

Tuned (best configuration) test accuracy: 0.9400
Exercise 16 hyperparameters test accuracy: 0.9300


**Did tuning help?**

Slightly. The tuned network (learning_rate ≈ 0.038, batch_size = 32, 200 epochs) reaches 0.94 on the test set, against 0.93 with the hyperparameters of Exercise 16. Over 100 test samples this is a difference of one sample, so the improvement is small and within the noise of the split.

The search results explain why: most configurations with a learning rate above ~0.001 reach a CV accuracy of 0.90–0.92, so the hyperparameters of Exercise 16 were already in the good region. The search mostly shows what does not work: very small learning rates (~0.0002–0.0007) stay at 0.71–0.77, because the network does not have enough steps to learn. Tuning matters most when the starting hyperparameters are far from a good region.